# LLM Structured Output using Pydentic

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B8F5869410>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B8F586B710>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

## Output without structure

In [4]:
response = model.invoke("Provide detailes of the move inception")
print(response)

content='<think>\nOkay, so I need to provide detailed information about the movie "Inception." Let me start by recalling what I know about it. Directed by Christopher Nolan, it\'s a sci-fi action thriller released in 2010. The main characters are Dom Cobb, played by Leonardo DiCaprio, and a team of experts who can enter people\'s dreams to plant ideas. The plot involves corporate espionage and the concept of shared dreaming. \n\nFirst, I should outline the basic plot. The story revolves around Cobb, a thief who steals information by infiltrating the subconscious of his targets. He is offered a chance to have his criminal history erased by performing the reverse: planting an idea into someone\'s mind, known as "inception." The target is a Japanese businessman, and the challenge is to make him invest in a new hotel. The team has to go through multiple layers of dreams, each deeper than the last, and there\'s a risk of getting stuck in the dream world if they don\'t wake up on time.\n\nI 

In [5]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description='The title of the movie')
    year:int = Field(description="This year the movie was released")
    director:str = Field(description='the director of the movie')
    rating:float = Field(description="the movie rating out of 10")

In [6]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B8F5869410>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B8F586B710>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'descri

## Output with structure

In [7]:
response = model_with_structure.invoke("Provide detailes of the move inception")
print(response)

title='Inception' year=2010 director='Christopher Nolan' rating=8.8


## Message output alongside parsed structure

In [9]:
class Movie(BaseModel):
    title:str = Field(...,description='The title of the movie')
    year:int = Field(...,description="This year the movie was released")
    director:str = Field(...,description='the director of the movie')
    rating:float = Field(...,description="the movie rating out of 10")

model_with_structure= model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("povide details about the movie Inception")
(response)

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check the tools available. There\'s a Movie function that requires title, year, director, and rating. I need to fill those parameters. I know Inception was directed by Christopher Nolan. The release year was 2010. The rating is a bit tricky; I think it\'s around 8.8 on IMDb, but the function wants a number out of 10. Let me confirm the exact rating. Oh, wait, the function might expect a numerical value without decimals. Maybe 8.8 is acceptable. Let me structure the JSON with the required fields. Title is "Inception", year 2010, director "Christopher Nolan", and rating 8.8. I should make sure all required parameters are included. Yep, that\'s covered. Alright, time to format the tool call correctly.\n', 'tool_calls': [{'id': 'd0c2jdg9w', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'nam

## Nested Structure

In [11]:
class Actor(BaseModel):
    name: str
    role: str

class movieDetails(BaseModel):
    title:str
    year: int
    cast: list[Actor]
    genres:list[str]
    budget:float|None=Field(None, description="Budget in millions USD")

    
model_with_structure= model.with_structured_output(movieDetails)

response = model_with_structure.invoke("povide details about the movie Inception")
response

movieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne')], genres=['Science Fiction', 'Action', 'Heist'], budget=160.0)

# TypedDict

In [12]:
from typing_extensions import TypedDict, Annotated 

class MovieDict(TypedDict):
    """A Movie with details"""
    title:Annotated[str, ..., "The title of the movie"]
    year:Annotated[int, ..., "the year the movie was released"]
    director:Annotated[str, ..., "the director of the movie"]
    rating:Annotated[float, ..., "The movie's ratting out of 10"]

    
model_with_structure= model.with_structured_output(MovieDict)

response = model_with_structure.invoke("povide details about the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [14]:
class Actor(TypedDict):
    name: str
    role: str

class movieDetails(TypedDict):
    title:str
    year: int
    cast: list[Actor]
    genres:list[str]
    budget:float|None=Field(None, description="Budget in millions USD")

    
model_with_structure= model.with_structured_output(movieDetails)

response = model_with_structure.invoke("povide details about the movie avengers")
response

{'budget': 220500000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Jude Law', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Science Fiction', 'Superhero'],
 'title': 'The Avengers',
 'year': 2012}

In [15]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

# Data Classes

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [20]:
result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='3715558f-771a-4f09-98be-ef72ce8727f4'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given string: "John Doe, john@example.com, (555) 123-4567". The tools provided include a ContactInfo function that requires name, email, and phone number.\n\nFirst, I need to parse the input. The name is John Doe. The email is clearly john@example.com. The phone number is (555) 123-4567. I should check if the phone number is in the correct format. The function parameters expect a string for each field. All required fields (name, email, phone) are present. No missing data here. So I can call the ContactInfo function with these details. Just need to make sure the JSON structure is correct with the right keys. No issues spotted. Ready to output the tool call

In [ ]:
class ContactInfo(TypedDict):
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

print(result["structured_response"])

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


In [22]:
from dataclasses import dataclass
@dataclass
class ContactInfo:
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

print(result["structured_response"])

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
